# Análise Exploratória de Dados

Análise de churn em empresa de telecomunicações.

Este notebook contém a análise exploratória inicial dos dados.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sys
import warnings
warnings.filterwarnings('ignore')

sys.path.append('..')

from src.data.load_data import generate_synthetic_data
from src.features.build_features import create_features
import yaml

In [ ]:
with open('../config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

In [ ]:
df = generate_synthetic_data(n_samples=5000)
print(f"Shape: {df.shape}")
df.head()

In [ ]:
print("=" * 50)
print("INFORMAÇÕES GERAIS")
print("=" * 50)
print(f"\nTotal de clientes: {len(df):,}")
print(f"Total de features: {len(df.columns)}")
print(f"\nChurn Rate: {df['churn'].mean():.2%}")
print(f"\nChurn: {df['churn'].sum():,} clientes")
print(f"Não Churn: {(df['churn'] == 0).sum():,} clientes")

In [ ]:
df.dtypes

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'Missing': missing, 'Percent': missing_pct}).query('Missing > 0')

In [ ]:
df.describe()

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Contagem', 'Percentual')
)

churn_counts = df['churn'].value_counts()

fig.add_trace(
    go.Bar(x=['Não Churn', 'Churn'], y=churn_counts.values,
           marker_color=['#2ecc71', '#e74c3c'], text=churn_counts.values),
    row=1, col=1
)

fig.add_trace(
    go.Pie(labels=['Não Churn', 'Churn'], values=churn_counts.values,
           marker_colors=['#2ecc71', '#e74c3c'], textinfo='percent+label'),
    row=1, col=2
)

fig.update_layout(title='Distribuição de Churn', showlegend=False, height=500)
fig.show()

In [ ]:
churn_by_contract = df.groupby('contract_type').agg({
    'churn': ['sum', 'count']
}).reset_index()
churn_by_contract.columns = ['contract_type', 'churn_count', 'total']
churn_by_contract['churn_rate'] = churn_by_contract['churn_count'] / churn_by_contract['total']
churn_by_contract = churn_by_contract.sort_values('churn_rate', ascending=False)

print("Churn Rate por Tipo de Contrato:")
print(churn_by_contract.to_string(index=False))

fig = px.bar(
    churn_by_contract, x='contract_type', y='churn_rate',
    text_auto='.1%', color='churn_rate', color_continuous_scale='RdYlGn_r',
    title='Taxa de Churn por Tipo de Contrato'
)
fig.update_layout(xaxis_title='Tipo de Contrato', yaxis_title='Taxa de Churn', showlegend=False)
fig.show()

In [ ]:
df_copy = df.copy()
df_copy['tenure_group'] = pd.cut(
    df_copy['tenure'],
    bins=[0, 6, 12, 24, 72],
    labels=['0-6 meses', '6-12 meses', '12-24 meses', '24+ meses']
)

churn_by_tenure = df_copy.groupby('tenure_group', observed=True).agg({
    'churn': ['sum', 'count']
}).reset_index()
churn_by_tenure.columns = ['tenure_group', 'churn_count', 'total']
churn_by_tenure['churn_rate'] = churn_by_tenure['churn_count'] / churn_by_tenure['total']

print("Churn Rate por Grupo de Tenure:")
print(churn_by_tenure.to_string(index=False))

fig = px.bar(
    churn_by_tenure, x='tenure_group', y='churn_rate',
    text_auto='.1%', color='churn_rate', color_continuous_scale='RdYlGn_r'
)
fig.update_layout(title='Taxa de Churn por Tenure', xaxis_title='Tenure', yaxis_title='Taxa de Churn', showlegend=False)
fig.show()

In [ ]:
churn_by_recl = df.groupby('num_reclamations_30d').agg({
    'churn': ['sum', 'count', 'mean']
}).reset_index()
churn_by_recl.columns = ['reclamations', 'churn_count', 'total', 'churn_rate']
churn_by_recl = churn_by_recl[churn_by_recl['reclamations'] <= 7]

print("Churn Rate por Número de Reclamações (30 dias):")
print(churn_by_recl.to_string(index=False))

high_recl = df[df['num_reclamations_30d'] >= 3]
print(f"\nClientes com 3+ reclamações: {high_recl['churn'].mean():.1%} churn rate")

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if 'customer_id' in numeric_cols:
    numeric_cols.remove('customer_id')

corr = df[numeric_cols].corr()

fig = px.imshow(
    corr, text_auto='.2f', color_continuous_scale='RdBu_r',
    title='Matriz de Correlação'
)
fig.update_layout(width=800, height=800)
fig.show()

In [ ]:
fig = px.histogram(
    df, x='monthly_charges', color='churn',
    nbins=40,
    title='Distribuição de Cobrança Mensal por Churn',
    barmode='overlay'
)
fig.update_layout(xaxis_title='Cobrança Mensal (R$)', yaxis_title='Frequência')
fig.show()

In [ ]:
churn_by_internet = df.groupby('internet_service').agg({
    'churn': ['sum', 'count', 'mean']
}).reset_index()
churn_by_internet.columns = ['internet_service', 'churn_count', 'total', 'churn_rate']

print("Churn Rate por Serviço de Internet:")
print(churn_by_internet.to_string(index=False))

fig = px.bar(
    churn_by_internet.sort_values('churn_rate', ascending=False),
    x='internet_service', y='churn_rate',
    text_auto='.1%', color='churn_rate', color_continuous_scale='OrRd'
)
fig.update_layout(title='Churn por Serviço de Internet', showlegend=False)
fig.show()

In [ ]:
print("=" * 60)
print("RESUMO DOS INSIGHTS")
print("=" * 60)

print("\n1. IMPACTO DE RECLAMAÇÕES:")
for i in range(5):
    recl_df = df[df['num_reclamations_30d'] == i]
    print(f"   {i} reclamações: {recl_df['churn'].mean():.1%} churn rate")

print("\n2. IMPACTO DO CONTRATO:")
for contract in df['contract_type'].unique():
    contract_df = df[df['contract_type'] == contract]
    print(f"   {contract}: {contract_df['churn'].mean():.1%} churn rate")

print("\n3. IMPACTO DO TENURE:")
tenure_bins = [(0, 6), (6, 12), (12, 24), (24, 72)]
for low, high in tenure_bins:
    tenure_df = df[(df['tenure'] >= low) & (df['tenure'] < high)]
    print(f"   {low}-{high} meses: {tenure_df['churn'].mean():.1%} churn rate")

In [ ]:
df_processed = create_features(df, config, fit=True)
df_processed.to_csv('../data/processed/telecom_churn_processed.csv', index=False)
print(f"Dados processados salvos! Shape: {df_processed.shape}")